# Diffusion Tensor Imaging Measurements for Computer-Aided Diagnosis of Autism (FAIR²) Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and processing the FAIR² Diffusion Tensor Imaging (DTI) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
This dataset follows the [Croissant](https://mlcommons.org/croissant/) standard. Its metadata (schema) is available at:

`https://sen.science/doi/10.71728/senscience.dbrh-5zc8/fair2.json`

In [ ]:
# Install mlcroissant if needed
!pip install mlcroissant

## 1. Data Loading
Load metadata and preview the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.dbrh-5zc8/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")

## 2. Data Overview
List available record sets, their `@id`s, and fields in the dataset. All references use the `@id` attribute as required by the Croissant standard.

In [ ]:
print("Available record sets in this dataset:")
record_set_ids = []

for record_set in dataset.metadata.record_sets:
    print(f"- @id: {record_set.id}, name: {record_set.name}")
    record_set_ids.append(record_set.id)
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - @id: {field.id}, name: {field.name}, type: {field.data_type}")

if not record_set_ids:
    print("No record sets found in Croissant schema.")

## 3. Data Extraction
Load data from all available record sets into DataFrames for further analysis.

We will use the record set and field `@id`s obtained above.

In [ ]:
# Identify available record sets
record_sets = record_set_ids
dataframes = {}

for record_set_id in record_sets:
    print(f"Loading records for record set {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f" - Loaded {len(df)} records, columns: {df.columns.tolist()}")
    else:
        print(f" - No records found for {record_set_id}.")

if dataframes:
    # Example: show the first 5 records of the first available record set
    preview_record_set_id = list(dataframes.keys())[0]
    print(f"\nPreview of the record set: {preview_record_set_id}")
    display(dataframes[preview_record_set_id].head())
else:
    print("No dataframes with records loaded. Please check dataset schema or Croissant specification.")

## 4. Exploratory Data Analysis (EDA)
Now we perform EDA on a chosen field and demonstrate common operations like filtering, normalizing, removing outliers, and grouping.

**Remember:** All field and record set references continue to use their `@id` values for consistency.

Below, pick a record set and numeric field you want to analyze. (This example assumes that at least one record set and numeric field exists.)

In [ ]:
# Example selection: use the first record set and first numeric field found
import numpy as np

selected_record_set_id = None
numeric_field_id = None
group_field_id = None

for record_set in dataset.metadata.record_sets:
    df = dataframes.get(record_set.id)
    if df is not None and len(df) > 0:
        for field in record_set.fields:
            # Use 'Float', 'Number', or 'Integer' as numeric
            if field.data_type in ["Float", "Number", "Integer"]:
                if field.id in df.columns:
                    selected_record_set_id = record_set.id
                    numeric_field_id = field.id
                    # Try to find a non-numeric field for grouping as example
                    for f2 in record_set.fields:
                        if f2.data_type in ["Text", "String"] and f2.id in df.columns:
                            group_field_id = f2.id
                            break
                    break
        if numeric_field_id:
            break

if not selected_record_set_id or not numeric_field_id:
    print("No suitable numeric field found for EDA. Please check the data schema.")
else:
    df = dataframes[selected_record_set_id]

    # Ensure column is numeric
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    mean_val = df[numeric_field_id].mean()
    std_val = df[numeric_field_id].std()

    # Remove outliers (> 3 std from mean)
    filtered_df = df[np.abs((df[numeric_field_id]-mean_val)/std_val) <= 3]
    print(f"After outlier removal, kept {len(filtered_df)} rows out of {len(df)}.")

    # Example filter: values above median
    threshold = df[numeric_field_id].median()
    filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]

    print(f"\nFiltered rows where {numeric_field_id} > {threshold} (median):\n")
    print(filtered_df[[numeric_field_id]].head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean_val) / std_val

    print(f"\nNormalized {numeric_field_id} for filtered rows:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].head()])

    # Grouping by group_field_id if available
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame("mean_value")
        print(f"\nGrouped by {group_field_id} (mean of {numeric_field_id}):")
        print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field, and (optionally) its relation to a group field, if any.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set_id and numeric_field_id:
    plt.figure(figsize=(8, 5))
    sns.histplot(data=filtered_df, x=numeric_field_id, kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id} (filtered)")
    plt.show()

    if group_field_id and group_field_id in filtered_df.columns:
        # Show a boxplot by group if reasonable
        plt.figure(figsize=(10,5))
        sns.boxplot(data=filtered_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=30, ha='right')
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we loaded the DTI measurements dataset (FAIR²) using `mlcroissant`, explored its record sets using `@id` references, performed basic EDA (including numeric transformations and group analysis), and visualized a key field.

- You may now proceed to use the loaded DataFrames for machine learning, further statistics, or domain-specific analysis.
- For more details, review the official Croissant schema [here](https://sen.science/doi/10.71728/senscience.dbrh-5zc8/fair2.json).

**Note:** Update the field `@id`s and EDA logic as desired to focus on relevant metrics or columns for your research.